# Notebook 6 — Final Evaluation and Comparison

**Project:** Equivariant Graph Neural Networks with Chirality Encoding for Structure-Sensitive Molecular Property Prediction

This notebook does **no model training** (except one optional controlled re-run in §6). It
loads the result files written by Notebooks 1–5 and assembles the final comparison:

> 2D GNN → 3D EGNN (conformer ensemble) → 3D + Stereo EGNN → Tox21 multi-task → BBBP

**Upload these files into the Colab session first** (Files panel):

| From | File |
|---|---|
| Notebook 1 | `results_2d_baseline.json` |
| Notebook 2 | `results_3d_egnn.json` |
| Notebook 3 | `results_3d_stereo_egnn.json`, `stereoisomer_discrimination_results.json` |
| Notebook 4 | `results_tox21.json` |
| Notebook 5 | `results_bbbp.json` |

Missing files are skipped gracefully, so you can run this notebook at any stage.

**A note on reading the results honestly.** Several comparisons below are *not* perfectly
controlled (different training-set sizes, different input features). Each section states
exactly what differs. Report the numbers together with those caveats — a viva examiner will
find the confounds if you don't, and naming them first is far more credible.


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110


def load_json(path):
    if not os.path.exists(path):
        print(f"[missing] {path}")
        return None
    with open(path) as f:
        return json.load(f)


r2d = load_json('results_2d_baseline.json')
r3d = load_json('results_3d_egnn.json')
rst = load_json('results_3d_stereo_egnn.json')
rdisc = load_json('stereoisomer_discrimination_results.json')
rtox = load_json('results_tox21.json')
rbbb = load_json('results_bbbp.json')

print("\nLoaded:", [n for n, r in [('2D', r2d), ('3D', r3d), ('Stereo', rst),
                                     ('Discrimination', rdisc), ('Tox21', rtox),
                                     ('BBBP', rbbb)] if r is not None])


## 1. QM9 regression: 2D → 3D → 3D + Stereo

Target: HOMO-LUMO gap (MAE in eV, lower is better).

**Confounds you must state when presenting this table:**
1. **Training-set size differs.** Notebook 1 trains on 16,000 molecules; Notebooks 2 and 3
   train on ~3,000 (RDKit conformer generation is expensive). A 2D model with 5x more data has
   a large built-in advantage.
2. **Input features differ.** Notebook 1 uses PyG's pre-computed 11-dim atom features
   (including hybridization and aromaticity); Notebooks 2–3 use bare atom-type one-hots plus
   3D coordinates.
3. **Test sets differ.** Notebook 1 evaluates on 2,000 test molecules; Notebooks 2–3 on ~500
   (a subset of the same split, minus any molecules that failed conformer generation).

So the 2D-vs-3D rows are **not** a clean measurement of "what 3D geometry adds". The 3D vs
3D+Stereo rows *are* comparable (same molecules, conformers, split and hyper-parameters), but
since `gap` is an achiral property we expect them to be close. §6 provides an optional
controlled re-run of the 2D baseline on the same subset.


In [ ]:
qm9_rows = []
for label, r in [('2D GNN baseline (NB1)', r2d),
                 ('3D EGNN + conformer ensemble (NB2)', r3d),
                 ('3D + Stereo EGNN (NB3)', rst)]:
    if r is None:
        continue
    n_train = r.get('num_train', r.get('num_train_molecules'))
    n_test = r.get('num_test', r.get('num_test_molecules'))
    qm9_rows.append({'model': label, 'mae_ev': r['test_mae_real_units'],
                     'n_train': n_train, 'n_test': n_test})

if qm9_rows:
    print(f"{'Model':38s} {'Test MAE (eV)':>14s} {'n_train':>8s} {'n_test':>7s}")
    print('-' * 72)
    for row in qm9_rows:
        print(f"{row['model']:38s} {row['mae_ev']:14.4f} {str(row['n_train']):>8s} {str(row['n_test']):>7s}")

    fig, ax = plt.subplots(figsize=(8, 3.8))
    names = [r['model'] for r in qm9_rows]
    maes = [r['mae_ev'] for r in qm9_rows]
    bars = ax.barh(names, maes, color=['tab:gray', 'tab:orange', 'tab:green'][:len(maes)])
    for bar, r in zip(bars, qm9_rows):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                f"  {r['mae_ev']:.3f} eV  (train n={r['n_train']})", va='center', fontsize=8)
    ax.set_xlabel('Test MAE, HOMO-LUMO gap (eV) — lower is better')
    ax.set_title('QM9 gap regression (NOT equal training sizes — see caveats)')
    ax.set_xlim(0, max(maes) * 1.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('fig_qm9_comparison.png', bbox_inches='tight')
    plt.show()
else:
    print("No QM9 result files found.")


## 2. Stereoisomer discrimination

For each curated pair of stereoisomers (same 2D graph, different 3D configuration) we compare
the L2 distance between the two isomers' embeddings under:
- the **stereo-blind** distance-only EGNN (untrained, by design — its invariance to mirror
  reflection is architectural, so any weights give ~0), and
- the **trained stereo-aware** encoder.

This is a **representation-level** test: it shows the encoder *can* separate stereoisomers.
It does **not** show improved prediction of any experimental property, because QM9's scalar
targets are identical for enantiomers.


In [ ]:
if rdisc:
    labels = [r['base_smiles'] for r in rdisc]
    blind = np.array([r['blind_embedding_l2_dist'] for r in rdisc])
    stereo = np.array([r['stereo_embedding_l2_dist'] for r in rdisc])

    print(f"{'Base SMILES':14s} {'blind L2':>12s} {'stereo L2':>12s}")
    print('-' * 42)
    for l, b, s in zip(labels, blind, stereo):
        print(f"{l:14s} {b:12.2e} {s:12.4f}")
    print(f"\nMean blind L2:  {blind.mean():.2e}")
    print(f"Mean stereo L2: {stereo.mean():.4f}")

    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.2, np.maximum(blind, 1e-9), 0.4, label='stereo-blind EGNN', color='tab:orange')
    ax.bar(x + 0.2, np.maximum(stereo, 1e-9), 0.4, label='stereo-aware EGNN', color='tab:green')
    ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('L2 distance between isomer embeddings (log scale)')
    ax.set_title('Stereoisomer discrimination (higher = better separation)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig_stereo_discrimination.png', bbox_inches='tight')
    plt.show()
else:
    print("No discrimination results found.")


## 3. Tox21 multi-task toxicity

Per-task ROC-AUC on the test set (random split). Notebook 4 has **no matched baselines** (no
GNN-only or stereo-blind Tox21 model was retrained), so this section reports absolute
performance only — it cannot say whether stereo or the fingerprint branch helped. The
fingerprint number from Notebook 4 is an inference-time probe on an already-trained model, not
a retrained ablation.

If you want a proper Tox21 ablation, the cleanest route is to reuse the `use_gnn` / `use_fp` /
`use_stereo` switches from Notebook 5's `BBBPModel` on a 12-output head.


In [ ]:
if rtox:
    per_task = rtox['per_task_roc_auc']
    names = list(per_task.keys())
    aucs = np.array([np.nan if v is None else v for v in per_task.values()], dtype=float)
    macro = rtox['test_macro_roc_auc']

    print(f"Test macro ROC-AUC: {macro:.4f}  (n_test = {rtox.get('num_test')})")
    if 'ablation_no_fp_auc' in rtox:
        print(f"Inference-time probe, fingerprints zeroed: {rtox['ablation_no_fp_auc']:.4f} "
              f"(difference {rtox['fp_branch_contribution']:+.4f})")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    colors = ['tab:red' if (not np.isnan(a) and a < 0.70) else 'tab:green' for a in aucs]
    ax.barh(names, np.nan_to_num(aucs), color=colors)
    ax.axvline(macro, color='steelblue', label=f'macro avg {macro:.3f}')
    ax.axvline(0.70, color='black', linestyle='--', linewidth=0.8, label='0.70')
    ax.set_xlim(0, 1)
    ax.set_xlabel('ROC-AUC')
    ax.set_title('Tox21 per-task test ROC-AUC')
    ax.legend(fontsize=8)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('fig_tox21_per_task.png', bbox_inches='tight')
    plt.show()
else:
    print("No Tox21 results found.")


## 4. BBBP ablation (scaffold split, mean ± std over seeds)

This is the only section with a **properly controlled ablation**: identical molecules,
conformers, scaffold split, hyper-parameters and seeds across all four variants. The helper
below flags whether differences exceed one standard deviation — differences smaller than that
should be described as "not distinguishable", not as wins.


In [ ]:
if rbbb:
    variants = rbbb['variants']
    order = [v for v in ['fp_only', 'gnn_no_stereo', 'gnn_stereo', 'hybrid'] if v in variants]

    print(f"{'Variant':16s} {'ROC-AUC':>16s} {'Accuracy':>16s} {'F1':>16s}")
    print('-' * 68)
    for v in order:
        r = variants[v]
        print(f"{v:16s} "
              f"{r['roc_auc']['mean']:.4f} ± {r['roc_auc']['std']:.4f}   "
              f"{r['accuracy']['mean']:.4f} ± {r['accuracy']['std']:.4f}   "
              f"{r['f1']['mean']:.4f} ± {r['f1']['std']:.4f}")

    def compare(a, b, metric='roc_auc'):
        if a not in variants or b not in variants:
            return
        da = variants[a][metric]['mean'] - variants[b][metric]['mean']
        noise = max(variants[a][metric]['std'], variants[b][metric]['std'])
        verdict = "exceeds 1 std" if abs(da) > noise else "within noise (not distinguishable)"
        print(f"  {a} vs {b}: {da:+.4f} AUC -> {verdict}")

    print("\nKey pairwise comparisons:")
    compare('gnn_stereo', 'gnn_no_stereo')     # does stereo help the GNN?
    compare('hybrid', 'fp_only')               # does the GNN add to fingerprints?
    compare('hybrid', 'gnn_stereo')            # do fingerprints add to the GNN?

    fig, ax = plt.subplots(figsize=(7, 4))
    means = [variants[v]['roc_auc']['mean'] for v in order]
    stds = [variants[v]['roc_auc']['std'] for v in order]
    ax.bar(order, means, yerr=stds, capsize=5,
           color=['tab:gray', 'tab:orange', 'tab:green', 'tab:blue'][:len(order)])
    ax.set_ylim(max(0.5, min(means) - 0.1), min(1.0, max(means) + 0.08))
    ax.set_ylabel('Test ROC-AUC (mean ± std)')
    ax.set_title(f"BBBP ablation — scaffold split, {len(rbbb.get('seeds', []))} seeds")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig('fig_bbbp_ablation.png', bbox_inches='tight')
    plt.show()
else:
    print("No BBBP results found.")


## 5. Consolidated summary


In [ ]:
summary = {}

if qm9_rows:
    summary['qm9_gap_mae_ev'] = {r['model']: {'mae_ev': r['mae_ev'], 'n_train': r['n_train'],
                                              'n_test': r['n_test']} for r in qm9_rows}
if rdisc:
    summary['stereo_discrimination'] = {
        'n_pairs': len(rdisc),
        'mean_blind_l2': float(np.mean([r['blind_embedding_l2_dist'] for r in rdisc])),
        'mean_stereo_l2': float(np.mean([r['stereo_embedding_l2_dist'] for r in rdisc])),
    }
if rtox:
    summary['tox21'] = {'macro_roc_auc': rtox['test_macro_roc_auc'],
                        'per_task_roc_auc': rtox['per_task_roc_auc']}
if rbbb:
    summary['bbbp'] = {v: {m: rbbb['variants'][v][m]['mean'] for m in ['roc_auc', 'accuracy', 'f1']}
                       for v in rbbb['variants']}

with open('final_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2, default=str))


## 6. (Optional) Controlled 2D baseline on the same subset as Notebooks 2–3

This closes confound #1 and #3 from §1 by training the Notebook 1 architecture on **the same
~3,000 / 500 / 500 molecule subset** that Notebooks 2–3 used (same seed and subsampling logic).
Confound #2 remains: this baseline still uses PyG's richer 11-dim atom features, whereas
Notebooks 2–3 use bare atom types — so if the 3D model still loses, part of that is feature
richness, not geometry.

Set `RUN_CONTROLLED_BASELINE = True` to run. Needs `qm9_split.pt` and `target_norm_stats.pt`
from Notebook 1, and takes a few minutes on a GPU. Note: Notebooks 2–3 silently dropped molecules
whose conformer generation failed, so the test sets match closely but not necessarily exactly.


In [ ]:
RUN_CONTROLLED_BASELINE = False

if RUN_CONTROLLED_BASELINE:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.datasets import QM9
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import NNConv, global_mean_pool

    SEED = 42
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    dataset = QM9(root='./data/QM9')
    split = torch.load('qm9_split.pt')
    norm = torch.load('target_norm_stats.pt')
    TARGET_IDX = split['target_idx']
    mean, std = norm['mean'], norm['std']

    # Same subsampling logic and generator order as Notebooks 2 and 3.
    g = torch.Generator().manual_seed(SEED)
    tr = split['train_idx'][torch.randperm(len(split['train_idx']), generator=g)[:3000]]
    va = split['val_idx'][torch.randperm(len(split['val_idx']), generator=g)[:500]]
    te = split['test_idx'][torch.randperm(len(split['test_idx']), generator=g)[:500]]

    class Wrap(torch.utils.data.Dataset):
        def __init__(self, idx):
            self.idx = idx
        def __len__(self):
            return len(self.idx)
        def __getitem__(self, i):
            d = dataset[self.idx[i]].clone()
            d.y = ((d.y[0, TARGET_IDX] - mean) / std).view(1)
            if hasattr(d, 'pos'):
                del d.pos
            return d

    tl = DataLoader(Wrap(tr), batch_size=128, shuffle=True)
    vl = DataLoader(Wrap(va), batch_size=256)
    tel = DataLoader(Wrap(te), batch_size=256)

    class GNN2D(nn.Module):
        def __init__(self, nin, ein, h=64, L=3):
            super().__init__()
            self.lin = nn.Linear(nin, h)
            self.convs = nn.ModuleList([
                NNConv(h, h, nn.Sequential(nn.Linear(ein, h), nn.ReLU(), nn.Linear(h, h * h)),
                       aggr='mean') for _ in range(L)])
            self.norms = nn.ModuleList([nn.BatchNorm1d(h) for _ in range(L)])
            self.out = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
        def forward(self, d):
            h = F.relu(self.lin(d.x))
            for c, n in zip(self.convs, self.norms):
                h = F.relu(n(c(h, d.edge_index, d.edge_attr)))
            return self.out(global_mean_pool(h, d.batch)).view(-1)

    model = GNN2D(dataset[0].x.size(-1), dataset[0].edge_attr.size(-1)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)

    @torch.no_grad()
    def mae_real(loader):
        model.eval()
        tot, n = 0.0, 0
        for d in loader:
            d = d.to(device)
            tot += (model(d) - d.y).abs().sum().item() * std.item()
            n += d.num_graphs
        return tot / n

    best_val, best_state = float('inf'), None
    for epoch in range(1, 61):
        model.train()
        for d in tl:
            d = d.to(device)
            opt.zero_grad()
            F.l1_loss(model(d), d.y).backward()
            opt.step()
        v = mae_real(vl)
        sched.step(v)
        if v < best_val:
            best_val = v
            best_state = {k: t.clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    controlled_mae = mae_real(tel)
    print(f"Controlled 2D baseline (same ~3000/500/500 subset) test MAE: {controlled_mae:.4f} eV")
    if r3d:
        print(f"3D EGNN ensemble (NB2): {r3d['test_mae_real_units']:.4f} eV")
    if rst:
        print(f"3D + Stereo EGNN (NB3): {rst['test_mae_real_units']:.4f} eV")
else:
    print("Skipped (set RUN_CONTROLLED_BASELINE = True to run).")


## 7. What to claim, and what not to claim

**Claims the evidence in this project can support:**
- A pipeline exists that takes a molecule from SMILES → low-energy RDKit conformers →
  equivariant 3D encoder → rotation/translation-invariant predictions (verified numerically in
  Notebooks 2–3).
- A distance-only EGNN is structurally blind to mirror images, while the stereo-aware encoder
  separates enantiomers and E/Z isomers (§2). This is a representation-capacity result.
- Masked BCE + positive weighting makes multi-task training on Tox21's sparse, imbalanced
  labels possible (§3).
- On BBBP, a controlled four-way ablation with scaffold splitting and multiple seeds quantifies
  what the GNN, stereo features and fingerprints each contribute (§4).

**Claims it does *not* support (unless your own numbers show otherwise):**
- *"3D geometry improved QM9 accuracy"* — only valid if the §6 controlled run shows it, and even
  then feature richness remains a confound.
- *"Stereo-awareness improved property prediction"* — QM9 `gap` is achiral, so no improvement is
  expected; BBBP depends mostly on polarity and size. A genuine test needs an endpoint where
  enantiomers differ experimentally (e.g. binding affinity or optical rotation data), which this
  project does not include. State this as **future work**.
- *"State-of-the-art"* — models here are small, trained on subsets, and Tox21 uses a random
  split, so numbers are not comparable to leaderboard results.

**Known limitations to mention proactively:**
1. Conformer generation used ETKDG + MMFF94 (force-field), not DFT; QM9 labels correspond to
   DFT-optimised geometries.
2. 3D models used ~3,000 training molecules (compute budget).
3. Chirality tags and signed volumes are precomputed at graph-construction time rather than
   recomputed inside the network.
4. Signed volume depends on a fixed neighbour ordering (by atom index); it is a parity-sensitive
   feature, not a full CIP-consistent descriptor.
5. Single conformer per molecule for Tox21 and BBBP; no conformer ensemble there.
6. None of the notebooks reproduce published baselines, so cross-paper comparison is indicative
   at best.

**Suggested future work:** train on all of QM9 with the 3D model for a fair comparison; add an
enantiomer-sensitive endpoint; scaffold-split Tox21; replace the precomputed chirality tags with
an end-to-end chirality-sensitive layer.
